# 03 - Anomaly Detection (LeakLens)
**Goal:** Find broken or unusual records, especially in campaigns.

**Approach:** (1) simple business rules for impossible data, then (2) one IsolationForest to catch *extra* suspicious rows the rules miss.

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest

DATA = Path("../data")
MODELS = Path("../models")
MODELS.mkdir(exist_ok=True)

def has_cols(df, cols):
    return all(c in df.columns for c in cols)

def to_bool(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y", "t"])

In [14]:
customers = pd.read_csv(DATA / "customers.csv")
products = pd.read_csv(DATA / "products.csv")
transactions = pd.read_csv(DATA / "transactions.csv")
inventory = pd.read_csv(DATA / "inventory.csv")
campaigns = pd.read_csv(DATA / "campaigns.csv")

for name, df in [("customers", customers), ("products", products),
                 ("transactions", transactions), ("inventory", inventory),
                 ("campaigns", campaigns)]:
    print(f"{name:13s} shape = {df.shape}")

customers     shape = (3000, 5)
products      shape = (250, 5)
transactions  shape = (18000, 10)
inventory     shape = (5000, 7)
campaigns     shape = (1200, 8)


In [15]:
rule_results = {}

# Rule 1: campaigns where conversions > clicks
campaigns["rule_conv_gt_clicks"] = False
if has_cols(campaigns, ["conversions", "clicks"]):
    campaigns["rule_conv_gt_clicks"] = campaigns["conversions"] > campaigns["clicks"]
rule_results["campaigns: conversions > clicks"] = int(campaigns["rule_conv_gt_clicks"].sum())

# Rule 2: products where unit_cost > list_price
products["rule_cost_gt_price"] = False
if has_cols(products, ["unit_cost", "list_price"]):
    products["rule_cost_gt_price"] = products["unit_cost"] > products["list_price"]
rule_results["products: unit_cost > list_price"] = int(products["rule_cost_gt_price"].sum())

# Rule 3: inventory where stockout_flag is true but stock_on_hand > 0
inventory["rule_stockout_but_stock"] = False
if has_cols(inventory, ["stockout_flag", "stock_on_hand"]):
    inventory["rule_stockout_but_stock"] = to_bool(inventory["stockout_flag"]) & (inventory["stock_on_hand"] > 0)
rule_results["inventory: stockout_flag but stock_on_hand > 0"] = int(inventory["rule_stockout_but_stock"].sum())

print("Rows caught by each rule:")
for rule, n in rule_results.items():
    print(f"  {rule:50s} {n}")

Rows caught by each rule:
  campaigns: conversions > clicks                    88
  products: unit_cost > list_price                   76
  inventory: stockout_flag but stock_on_hand > 0     760


In [16]:
def safe_div(a, b):
    return np.where(b > 0, a / np.where(b > 0, b, 1), np.nan)

base_cols = [c for c in ["spend", "impressions", "clicks", "conversions", "revenue"] if c in campaigns.columns]
feat_df = campaigns[base_cols].apply(pd.to_numeric, errors="coerce").copy()

# Ratio features (only if their inputs exist)
if has_cols(feat_df, ["clicks", "impressions"]):
    feat_df["ctr"] = safe_div(feat_df["clicks"], feat_df["impressions"])
if has_cols(feat_df, ["conversions", "clicks"]):
    feat_df["conv_rate"] = safe_div(feat_df["conversions"], feat_df["clicks"])
if has_cols(feat_df, ["spend", "clicks"]):
    feat_df["cost_per_click"] = safe_div(feat_df["spend"], feat_df["clicks"])
if has_cols(feat_df, ["spend", "conversions"]):
    feat_df["cost_per_conversion"] = safe_div(feat_df["spend"], feat_df["conversions"])

feature_list = list(feat_df.columns)
fill_values = feat_df.median(numeric_only=True).to_dict()
X = feat_df.fillna(fill_values)

print("Features used:", feature_list)
print("Feature matrix shape:", X.shape)

Features used: ['spend', 'impressions', 'clicks', 'conversions', 'ctr', 'conv_rate', 'cost_per_click', 'cost_per_conversion']
Feature matrix shape: (1200, 8)


In [17]:
CONTAMINATION = 0.03   # expect ~3% of campaigns to be unusual

iso = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=42)
iso.fit(X)

campaigns["iso_flag"] = iso.predict(X) == -1
campaigns["anomaly_score"] = -iso.score_samples(X)   # higher = more unusual
print("Trained IsolationForest on shape:", X.shape)

Trained IsolationForest on shape: (1200, 8)


In [18]:
rule_flag = campaigns["rule_conv_gt_clicks"]
iso_flag = campaigns["iso_flag"]
extra_flag = iso_flag & ~rule_flag

print(f"Total campaigns:                          {len(campaigns)}")
print(f"Caught by rules (conversions > clicks):   {int(rule_flag.sum())}")
print(f"Flagged by IsolationForest:               {int(iso_flag.sum())}")
print(f"  of which also caught by rules:          {int((iso_flag & rule_flag).sum())}")
print(f"EXTRA rows found only by IsolationForest: {int(extra_flag.sum())}")
print(f"Total unique suspicious campaigns:        {int((rule_flag | iso_flag).sum())}")

id_cols = [c for c in ["campaign_id", "date", "channel", "category"] if c in campaigns.columns]
# Feature columns (incl. ratios like ctr) live in X, not in campaigns, so join them for display
view = pd.concat([campaigns[id_cols], X.round(3), campaigns[["anomaly_score"]].round(3)], axis=1)
print("\nTop 10 extra anomalies (not caught by rules):")
display(view.loc[extra_flag].sort_values("anomaly_score", ascending=False).head(10))

Total campaigns:                          1200
Caught by rules (conversions > clicks):   88
Flagged by IsolationForest:               36
  of which also caught by rules:          16
EXTRA rows found only by IsolationForest: 20
Total unique suspicious campaigns:        108

Top 10 extra anomalies (not caught by rules):


,campaign_id,date,channel,category,spend,impressions,clicks,conversions,ctr,conv_rate,cost_per_click,cost_per_conversion,anomaly_score
1090,CA1091,2025-07-19,App,Home,7254.14,117,6463,1325,55.239,0.205,1.122,5.475,0.646
207,CA0208,2025-11-14,Social,Home,640.03,395,9546,390,24.167,0.041,0.067,1.641,0.625
234,CA0235,2025-05-03,Store,Grocery,4224.98,26605,9918,4,0.373,0.000,0.426,1056.245,0.623
1089,CA1090,2025-02-03,App,Home,3499.77,117,6924,772,59.179,0.111,0.505,4.533,0.623
781,CA0782,2025-10-17,Web,Sports,7238.42,30226,8137,8,0.269,0.001,0.890,904.802,0.620
158,CA0159,2025-10-15,Web,Sports,1880.64,269,9351,1165,34.762,0.125,0.201,1.614,0.617
452,CA0453,2025-10-07,Store,Fashion,5938.64,67622,3941,3,0.058,0.001,1.507,1979.547,0.609
747,CA0748,2025-12-04,Social,Fashion,2719.71,123,3538,872,28.764,0.246,0.769,3.119,0.603
156,CA0157,2025-02-16,Social,Fashion,7616.79,28622,9030,21,0.315,0.002,0.843,362.704,0.597
1020,CA1021,2025-11-29,Social,Fashion,3398.22,258,5619,1256,21.779,0.224,0.605,2.706,0.592


In [19]:
artifact = {
    "model": iso,
    "feature_list": feature_list,
    "fill_values": fill_values,
    "contamination": CONTAMINATION,
    "rules": list(rule_results.keys()),
}
joblib.dump(artifact, MODELS / "anomaly_model.joblib")
print("Saved:", MODELS / "anomaly_model.joblib")

Saved: ..\models\anomaly_model.joblib


## Summary for judges
- **Two layers of protection:** transparent business rules catch provably impossible records; one IsolationForest finds *unusual-but-possible* campaigns the rules miss.
- **Explainable:** every rule is a one-line check, and each flagged row has an anomaly score for ranking.
- **Business value:** broken campaign data inflates reported performance. Flagging it stops LeakLens from recommending actions based on bad numbers.
- **Reusable:** the saved model includes its feature list and fill values, so new campaigns can be scored the same way.